# Load Libraries

In [25]:
from pathlib import Path
import importlib.util
from joblib import Parallel, delayed
import os

def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "functions").is_dir() and (candidate / "pyspedas").is_dir():
            return candidate
    raise RuntimeError("Could not locate MHDTurbPy root (missing functions/ and pyspedas/).")


root_dir = _find_repo_root(Path.cwd())
path_setup_file = root_dir / "functions" / "path_setup.py"
spec = importlib.util.spec_from_file_location("mhdturbpy_path_setup", path_setup_file)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load path setup from {path_setup_file}")
path_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(path_setup)

root_dir = path_setup.ensure_project_paths(
    start=Path.cwd(),
    include_downloading_helpers=True,
    include_anisotropy_toolbox=True,
    include_sc_pos=True,
)


from functions import download_data as download

from functions import  calc_diagnostics as calc
from functions import  TurbPy as turb
from functions import general_functions as func
from functions import  Figures as figs
from functions import  interactive_figs

# Doawnload data

In [24]:
"""
mre_run_solo.py

Minimal-but-complete MRE to run the SOLO pipeline.

RUN FROM REPO ROOT (the directory that contains: pyspedas/, functions/, download_data.py)
"""







# ---------------------------------------------------------------------
# CDFlib path (needed for SOLO distance download via CDAS)
# ---------------------------------------------------------------------
cdf_lib_path = "/Applications/cdf/cdf/lib"


# ---------------------------------------------------------------------
# Credentials (None is fine for SOLO in most cases)
# ---------------------------------------------------------------------
credentials = None


# How many cores to use
n_jobs     = 1

# ---------------------------------------------------------------------
# Settings (organized by topic, then flattened into one dict)
# ---------------------------------------------------------------------
Paths = {
    
        "Data_path"         : Path(root_dir).joinpath('data'),
        "save_destination"  :  Path(root_dir).joinpath('examples').joinpath('downloaded_intervals'),
        #"SOLO_dist_path"    : "/Users/turbulator/work/turb_amplitudes/SOLO/solo_dist.pkl",
}

Mission = { 
    "sc"             : "SOLO",
    "in_rtn"         : True,
    "use_local_data" : False,
}

Intervals = {
    # "start_date": "2025-01-10 00:00",
    # "end_date": "2025-01-10 06:00",

    "start_date": "2022-10-01 00:00",
    "end_date": "2025-10-03 00:00",
    
    "multiple_intervals": True,   # False -> single interval, True -> use Step/duration below
    "duration": "30d",
    "Step": "30d",
    "addit_time_around": 1,        # hours padding in LoadTimeSeriesSOLO
}

Sampling = {
    "part_resol": 1000,
    "MAG_resol": 1000,
    "upsample_low_freq_ts": False,
}

QualityControl = {
    "overwrite_files"   :  0,
    "must_have_qtn": False,
    "Max_par_missing": 30,
    "gap_time_threshold": 5,
    "use_hampel": False,
    "hampel_params": {"w": 200, "std": 3},
}

Output = {
    
    "save_all": True,
    "cut_in_small_windows": {
        "flag": False,
        "njobs": 1,
        "Step": "5s",
        "duration": "30s",
    },
}

Diagnostics = {
    "estimate_derived_param": True,
    "rol_window": "60min",

    "PSDs": {"flag": False},
    "struc_funcs": {"flag": False},

    "npt_struc_funcs": {
        "flag": False,
        "five_points_sfunc": True,
        "return_Bmod": True,
        "dt_step": 0.25,
        "est_sfuncs": False,
        "max_qorder": 8,
    },

    "estimate_psd_b": True,
    "estimate_psd_v": True,
    "est_PSD_components": True,
    "smooth_psd": False,
}

Gaps = {
    "Big_Gaps": {
        "E_big_gaps": 10,
        "SC_pot_big_gaps": 10,
        "Mag_big_gaps": 500,
        "Par_big_gaps": 500,
        "QTN_big_gaps": 10,
    }
}

SOLO = {
    "SOLO_use_merged_MAG": False,   # set False to use SPEDAS L2 MAG
    "SOLO_merged_fs": 256,         # 256 or 4096
}

# ---- Unified MAG noise removal (works for SOLO merged MAG and PSP SCAM)
# This is the preferred key; legacy Mag_SCAM_PSP["noise_flag"] still works.
MAG_NOISE = {
    "Mag_SCM": {
        "noise_flag": False,
        "noise_removal": {
            "freq_min": 8.0,

            "stft_nperseg": 2048,
            "stft_overlap": 0.5,

            "percentile_q": 99.5,
            "kernel": 301,
            "thresh_db": 3.0,
            "merge_hz": 0.20,
            "max_lines": 2000,

            "track_half_width_hz": 5.0,
            "remove_half_width_hz": 1.5,
            "atten_db": 100.0,

            "fallback_top_k": 120,
            "whiten_exp": 0.0,
        },
    }
}







# ---- final settings dict (what pipeline consumes)
settings = {
    **Paths,
    **Mission,
    **Intervals,
    **Sampling,
    **QualityControl,
    **Output,
    **Diagnostics,
    **SOLO,
    **MAG_NOISE,
    "Big_Gaps": Gaps["Big_Gaps"],
}


# ---------------------------------------------------------------------
# What variables to download (None -> defaults inside SOLO.py)
# ---------------------------------------------------------------------
vars_2_downnload = {
    "mag": None,
    "swa": None,
    "rpw": None,
    "ephem": None,
}


# ---------------------------------------------------------------------
# Interval generation
# ---------------------------------------------------------------------
generated_interval_list = download.generate_intervals(
    settings["start_date"],
    settings["end_date"],
    settings["multiple_intervals"],
    data_path=settings["Data_path"],
    settings=settings,
)


# ---------------------------------------------------------------------
# Run download for each interval
# ---------------------------------------------------------------------
os.chdir(settings["Data_path"])

save_path = Path(settings["save_destination"]).joinpath(settings["sc"])
save_path.mkdir(parents=True, exist_ok=True)

Parallel(n_jobs=n_jobs)(
    delayed(download.download_files)(
        jj,
        generated_interval_list,
        settings,
        vars_2_downnload,
        cdf_lib_path,
        credentials,
        save_path,
    )
    for jj in range(len(generated_interval_list))
)


17-Feb-26 05:37:56: Generated intervals with Step: 30d and Duration: 30d
17-Feb-26 05:37:56: Number of Intervals: 36
Start Time: 2022-10-01 00:00:00
17-Feb-26 05:37:56: End Time: 2025-10-03 00:00:00


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [ ]:
,

In [11]:
#  User defined parameters
sc          = 'SOLO'
which_int   = 0
load_path   = Path(root_dir).joinpath('examples').joinpath('downloaded_intervals').joinpath(sc)


# Locate the downloaded files
finnames      = func.load_files(load_path, 'final.pkl')
gennames      = func.load_files(load_path, 'general.pkl')
signames      = func.load_files(load_path, 'sig_c_sig_r.pkl')
maggaps       = func.load_files(load_path, 'mag_gaps.pkl')
qtngaps       = func.load_files(load_path, 'qtn_gaps.pkl')
pargaps       = func.load_files(load_path, 'par_gaps.pkl')
sc_pot        = func.load_files(load_path, 'sc_pot_gaps.pkl')


#Load the filed
fin         = pd.read_pickle(finnames[which_int])
gen         = pd.read_pickle(gennames[which_int])
sig         = pd.read_pickle(signames[which_int])
mag_gaps    = pd.read_pickle(maggaps[which_int])
qtn_gaps    = pd.read_pickle(qtngaps[which_int])
par_gaps    = pd.read_pickle(pargaps[which_int])
sc_pot_gaps = pd.read_pickle(sc_pot[which_int])

finnames[which_int]

NameError: name 'func' is not defined

In [32]:
import plotly.io as pio
print("plotly renderer =", pio.renderers.default)


plotly renderer = plotly_mimetype


In [31]:
# user defined parameters

label_size        = 21                                    # labels etc

n_subplots        = 7                                     # number os subplots
my_dir            = '/Users/nokni/work/MHDTurbPy/examples/' #
format_2_return   = "%Y_%m_%d"                            # Format to save figures


figs.visualize_downloaded_intervals(
                                  sc                         ,
                                  fin['Par']['V_resampled'],
                                  fin['Mag']['B_resampled'],
                                  sig     ,
                                  my_dir,
                                  format_2_return  = "%Y_%m_%d",  #
                                  size             = label_size,
                                  numb_subplots    = n_subplots
                                 )



# Run with interactive figure

In [5]:
%matplotlib tk

from pathlib import Path
import matplotlib.pyplot as plt
from functions import interactive_figs
from functions import general_functions as func

plt.close("all")

sc = "SOLO"
which_int = 0
load_path = r"C:\Users\nokni\work\MHDTurbPy\examples\SOLO"
my_dir   = r"C:\Users\nokni\work\MHDTurbPy\examples"
save_path = Path(my_dir) / "selected_intervals"

# ============================================================
# ONE run that does BOTH:
# (A) add one extra curve to an existing panel (panel 0)
# (B) append a brand-new extra panel (8th subplot)
# ============================================================
panel_edits = {
    # (A) add one extra curve to an existing panel
    "add_series": {
        "panel_idx": 0,   # top panel
        "axis_idx": 0,    # first axis-spec in that panel (left axis)
        "series": {
            "kind": "col",
            "col": "B_RTN",
            "label": r"$|B|$ (extra dashed)",
            "style": {"lw": 1.0, "ls": "--", "ms": 0, "color": "0.35"},
        },
    },

    # (B) append a new panel at the bottom
    "add_panels": {
        "where": "append",
        "panel": {
            "axes": [
                {
                    "axis_id": "left",
                    "source": "Par",
                    "scale": "linear",
                    "series": [
                        {
                            "kind": "col",
                            "col": "Vth",
                            "label": r"$T_p~[eV]$ (extra panel)",
                            "style": {"lw": 0.8, "ls": "-", "ms": 0, "color": "k"},
                        }
                    ],
                    "legend": {"fontsize": "small", "frameon": False, "bbox_to_anchor": (1.01, 1), "loc": 2},
                    "hline": [{"y": 100.0, "ls": ":", "c": "0.3", "lw": 1.2}],
                }
            ]
        },
    },

    # (c) More

    
}

fig, events = interactive_figs.interactive_mhdturbpy_interval(
    sc=sc,
    which_int=which_int,
    load_path=load_path,
    my_dir=my_dir,
    save_path=save_path,
    rolling=None,
    resample_rule=None,
    fill_method=None,
    gap_thresholds={"mag": "30s", "par": "30s", "qtn": "30s", "sc_pot": "30s"},
    load_files_func=func.load_files,
    autosave=True,
    resume=True,
    export_csv=True,
    snap_to_data=False,
    enable_comments=True,
    debug_interaction=True,
    panel_config=None,          # use defaults
    panel_edits=panel_edits,    # add curve + append panel
    auto_ylims=True,
    debug_plot_config=True,     # should print n_panels=8
)

plt.show()


[plot] n_panels=8
[picker] resumed 3 intervals from C:\Users\nokni\work\MHDTurbPy\examples\selected_intervals\selected_intervals_2024_10_10_2024_10_10_SOLO.pkl
installed mpl(cids=16,None,17,18) + tkbinds=3MHDTurbPy\examples\selected_intervals\selected_intervals_2024_10_10_2024_10_10_SOLO.pkl

[picker] left_select_fallback(default)=True  (toggle with 'l')s\selected_intervals\selected_intervals_2024_10_10_2024_10_10_SOLO.pkl


dedupe_ms=250  dedupe_tol_ns=5000000  span_axes=8 marker_axes=10  move_throttle_ms=33  use_release_event=False0_10_SOLO.pkl





In [13]:
# ---------- User configuration ----------
cdf_lib_path = "/Applications/cdf/cdf/lib"  # update for your machine if needed
credentials = None

# Scan window (end is exclusive for daily slicing)
start_date = "2022-10-01 00:00"
end_date   = "2025-11-01 00:00"

# SOAR merged product options
SOLO_merged_fs = 256  # 256 or 4096
in_rtn = 1            # 1 -> RTN product, 0 -> SRF product

# Save options
save_usual_files = True
save_base = Path(root_dir) / "examples" / "downloaded_intervals" / "SOLO_merged_MAG_daily"
save_base.mkdir(parents=True, exist_ok=True)

settings = {
    "Data_path": Path(root_dir) / "data",
    "save_destination": Path(root_dir) / "examples" / "downloaded_intervals",
    "sc": "SOLO",
    "in_rtn": in_rtn,
    "use_local_data": False,

    # daily non-overlapping intervals
    "start_date": start_date,
    "end_date": end_date,
    "multiple_intervals": False,
    "duration": "24H",
    "Step": "24H",
    "addit_time_around": 1,

    # 1-minute products
    "part_resol": 60,
    "MAG_resol": 60,
    "upsample_low_freq_ts": False,

    "overwrite_files": 1,
    "must_have_qtn": False,
    "Max_par_missing": 30,
    "gap_time_threshold": 5,
    "save_all": True,

    "estimate_derived_param": True,
    "rol_window": "60min",

    # Required by your request
    "SOLO_use_merged_MAG": True,
    "SOLO_merged_fs": SOLO_merged_fs,

    "Big_Gaps": {
        "E_big_gaps": 10,
        "SC_pot_big_gaps": 10,
        "Mag_big_gaps": 500,
        "Par_big_gaps": 500,
        "QTN_big_gaps": 10,
    },

    "cut_in_small_windows": {
        "flag": False,
        "njobs": 1,
        "Step": "5s",
        "duration": "30s",
    },
}

vars_2_downnload = {"mag": None, "swa": None, "rpw": None, "ephem": None}

frame = "rtn" if in_rtn else "srf"
merged_product = f"multi-mag-rpw-scm-merged-{frame}-{SOLO_merged_fs}"
settings["SOLO_merged_product"] = merged_product

print("Save path:", save_base)
print("Merged product:", merged_product)

     


Save path: C:\Users\nokni\work\MHDTurbPy\examples\downloaded_intervals\SOLO_merged_MAG_daily
Merged product: multi-mag-rpw-scm-merged-rtn-256


In [14]:
import pandas as pd

# Daily non-overlapping intervals
start_ts = pd.Timestamp(start_date)
end_ts = pd.Timestamp(end_date)

edges = pd.date_range(start=start_ts, end=end_ts, freq="1D")
if len(edges) < 2:
    raise ValueError("Need at least 1 full day in [start_date, end_date].")

intervals = pd.DataFrame({"Start": edges[:-1], "End": edges[1:]})
print(f"Generated {len(intervals)} daily intervals.")
intervals.head()

Generated 1127 daily intervals.


,Start,End
0,2022-10-01,2022-10-02
1,2022-10-02,2022-10-03
2,2022-10-03,2022-10-04
3,2022-10-04,2022-10-05
4,2022-10-05,2022-10-06


In [15]:
def precheck_soar_availability(interval_df, product):
    rows = []
    try:
        from sunpy.net import Fido
        import sunpy.net.attrs as a
        import sunpy_soar  # noqa: F401
        soar_ok = True
    except Exception as e:
        print("SOAR metadata pre-check unavailable:", e)
        print("Falling back to full-run candidates for all days.")
        soar_ok = False

    if not soar_ok:
        for _, r in interval_df.iterrows():
            rows.append({
                "Start": r["Start"],
                "End": r["End"],
                "soar_available": 1,
                "soar_n_records": np.nan,
            })
        return pd.DataFrame(rows)

    for _, r in interval_df.iterrows():
        t0 = pd.Timestamp(r["Start"])
        t1 = pd.Timestamp(r["End"])

        try:
            qr = Fido.search(a.Time(t0, t1), a.soar.Product(product))
            nrec = len(qr)
            available = int(nrec > 0)
        except Exception:
            nrec = np.nan
            available = 0

        rows.append({
            "Start": t0,
            "End": t1,
            "soar_available": available,
            "soar_n_records": nrec,
        })

    return pd.DataFrame(rows)


precheck = precheck_soar_availability(intervals, merged_product)
print(
    f"SOAR candidate days: {int(precheck.soar_available.sum())}/{len(precheck)}"
)
precheck.head(20)

     

SOAR candidate days: 1127/1127


,Start,End,soar_available,soar_n_records
0,2022-10-01,2022-10-02,1,1
1,2022-10-02,2022-10-03,1,1
2,2022-10-03,2022-10-04,1,1
3,2022-10-04,2022-10-05,1,1
4,2022-10-05,2022-10-06,1,1
5,2022-10-06,2022-10-07,1,1
6,2022-10-07,2022-10-08,1,1
7,2022-10-08,2022-10-09,1,1
8,2022-10-09,2022-10-10,1,1
9,2022-10-10,2022-10-11,1,1


In [16]:
precheck

,Start,End,soar_available,soar_n_records
0,2022-10-01,2022-10-02,1,1
1,2022-10-02,2022-10-03,1,1
2,2022-10-03,2022-10-04,1,1
3,2022-10-04,2022-10-05,1,1
4,2022-10-05,2022-10-06,1,1
...,...,...,...,...
1122,2025-10-27,2025-10-28,1,1
1123,2025-10-28,2025-10-29,1,1
1124,2025-10-29,2025-10-30,1,1
1125,2025-10-30,2025-10-31,1,1


In [29]:
# run_interactive_interval.py
#
# Minimal driver for MHDTurbPy interval visualization.

from importlib import reload
from pathlib import Path

import matplotlib
matplotlib.use("TkAgg")  # notebook: use %matplotlib tk

import matplotlib.pyplot as plt

import interactive_figs as ifigs
import general_functions as func

reload(ifigs)
plt.close("all")

ROOT = Path(r"C:\Users\nokni\work\MHDTurbPy")  # <-- edit this once

SC = ["SOLO"]  # sc[0] used by flow_mode="sc1_v", sc[1] by "sc2_v"

cfg = dict(
    sc=list(SC),
    which_int=19,
    load_path={s: ROOT / "examples" / "downloaded_intervals" / s for s in SC},
    my_dir=ROOT / "examples",
    save_path=(ROOT / "examples" / "selected_intervals"),
    load_files_func=func.load_files,

    rolling="30min",
    align_intervals_to_first_sc=True,

    enable_flow_separation=False,
    flow_mode="sc1_v",
    flow_v_smooth_window="2min",
    flow_dir_gse=(-1.0, 0.0, 0.0),  # fallback only (used if V is missing/invalid)
    vsw_fallback=400.0,

    alternate_legend_sides=True,
)

cfg["save_path"].mkdir(parents=True, exist_ok=True)

fig, events = ifigs.interactive_mhdturbpy_interval(**cfg)
plt.show()


In [ ]:
What will this produce? Is it integrated with the download_data pipeline? Such that this a data product added to final['Par']['V_resampled'] at the cadence of the original timesries? Can we do that? THINK DEEPLYA ND REVSIE! AS ususal, make as few changes as possibl